# Photo App: End-to-End

All the pieces are built. We have a Flet desktop frontend, a FastAPI backend with photo, search, people, summaries, and pipeline routers, a Postgres database with pgvector, an S3 indexing pipeline with CLIP and FaceNet, WebSocket progress streaming, and GPT-4o-mini summaries. This notebook assembles them into a single running system. We trace each of the three main user actions through the full stack, address the performance and reliability concerns that surface at scale, and close with a map of how the architecture grows when the app outgrows a single machine.

## System Overview

**Components.** The production-local stack consists of four runtime components:

- **Flet desktop app:** runs on the host machine; the user interface. Communicates with the API over HTTP and WebSocket.
- **FastAPI API server:** runs in a Docker container, listening on `localhost:8000`. Handles all business logic.
- **Postgres + pgvector:** runs in a Docker container, reachable from the API container over the Docker Compose internal network as `db:5432`. Stores photo metadata, face records, embeddings, and summaries.
- **S3:** remote object storage. The API reads and writes directly; presigned URLs let the Flet app fetch images without proxying through the API.

The API and database containers are defined in `docker-compose.yml` (see notebook 06). Flet runs on the host (not in Docker) because it opens a native desktop window.

<br>

**Data flow for the three main user actions:**

1. **Browse.** The Flet gallery calls `GET /photos/?page=N` → the API queries Postgres for a page of photos sorted by `taken_at DESC` → for each photo, the API generates a presigned S3 URL → the photo list with URLs is returned as JSON → Flet renders each URL as an `ft.Image` control. The presigned URLs are generated in a tight Python loop (no I/O), so the entire response takes roughly the time of one Postgres query plus serialization.

2. **Search.** The Flet search bar sends `POST /search/` with a query string → the API encodes the query with the CLIP text encoder (≈20ms on CPU) → pgvector executes an HNSW approximate nearest-neighbor search over the `embedding` column (≈5ms for 100K photos) → the top-$k$ photo IDs are returned → the API batch-generates presigned URLs → JSON response to Flet.

3. **Index.** Flet triggers `POST /pipeline/start` → the API launches a `BackgroundTask` and returns a `job_id` immediately → the background task scans the S3 prefix, downloads each photo, extracts EXIF, runs CLIP and FaceNet, upserts all records into Postgres → after each batch it calls `manager.broadcast(job_id, event.model_dump_json())` → the Flet `IndexingProgress` component, subscribed to `GET /pipeline/{job_id}/ws`, receives each event and updates the progress bar live.

:::{.callout-note}
`CORSMiddleware` must allow `http://localhost:*` because the Flet desktop app embeds a Chromium webview whose origin is `localhost` with a randomly assigned port. Without CORS headers the browser will block all API requests.

:::

## Full Stack Wiring

**`main.py`.** The FastAPI application is assembled in a single entry point: all routers are included, the startup lifespan connects to the database and runs pending Alembic migrations, and CORS is configured. We use the `lifespan` context manager (introduced in Starlette 0.20) rather than the deprecated `@app.on_event` hooks, which makes startup and shutdown logic easier to test.

<br>

**Alembic on startup.** Running `alembic upgrade head` automatically at startup ensures the schema is always in sync with the codebase: a new deployment or a fresh database will have the correct tables without a manual migration step. The trade-off is a small startup delay (< 1s for a few dozen migrations). For production deployments, a pre-start migration job is preferable; for local development, auto-upgrade is convenient.

Minimal `main.py` with all routers and lifespan:

In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware


# --- Stub router imports (would be real modules in production) ---
from fastapi import APIRouter

photos_router    = APIRouter(prefix="/photos",    tags=["photos"])
search_router    = APIRouter(prefix="/search",    tags=["search"])
people_router    = APIRouter(prefix="/people",    tags=["people"])
summaries_router = APIRouter(prefix="/summaries", tags=["summaries"])
pipeline_router  = APIRouter(prefix="/pipeline",  tags=["pipeline"])
timeline_router  = APIRouter(prefix="/timeline",  tags=["timeline"])


# --- Lifespan ---
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: connect DB, run migrations
    print("[startup] connecting to database...")
    # await database.connect()
    # subprocess.run(["alembic", "upgrade", "head"], check=True)
    print("[startup] migrations applied")
    yield
    # Shutdown: close DB pool, drain WebSocket manager
    print("[shutdown] closing database pool...")
    # await database.disconnect()


# --- App assembly ---
app = FastAPI(title="Photo App", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost", "http://localhost:8550"],  # <1>
    allow_origin_regex=r"http://localhost:\d+",                   # <2>
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

for router in (photos_router, search_router, people_router,
               summaries_router, pipeline_router, timeline_router):
    app.include_router(router)


@app.get("/health")
async def health() -> dict:
    return {"status": "ok"}


print(f"Registered routes: {[r.path for r in app.routes]}")

1. Explicit origins for Flet's default port.
2. Regex catch-all for any `localhost` port: Flet assigns the webview port dynamically.

## Timeline View

**Grouping photos by month.** The timeline view is the app's home screen: a vertical list of month cards, each showing the month name, the photo count, and a representative cover photo. We compute this with a single SQL aggregation — `date_trunc('month', taken_at)` groups photos into calendar months; `COUNT(*)` gives the total; and a `DISTINCT ON` or subquery picks the cover photo (the one with the highest CLIP similarity to the month label text, or simply the most recent).

<br>

**`MonthGroup` schema.** The API returns a `list[MonthGroup]` ordered newest-first. The Flet timeline scrolls through these cards; tapping a card navigates to a filtered gallery showing only that month's photos.

Defining the timeline schemas and `GET /timeline/` endpoint:

In [ ]:
from datetime import date
from pydantic import BaseModel
from fastapi import APIRouter
from fastapi.testclient import TestClient
from fastapi import FastAPI


class MonthGroup(BaseModel):
    month:           date          # first day of the month
    count:           int
    cover_photo_url: str | None    # presigned S3 URL


# Stub: in production this runs a SQLAlchemy query with func.date_trunc
MOCK_TIMELINE = [
    MonthGroup(month=date(2023, 8, 1), count=47, cover_photo_url="https://s3.example.com/p1.jpg"),
    MonthGroup(month=date(2023, 7, 1), count=112, cover_photo_url="https://s3.example.com/p2.jpg"),
    MonthGroup(month=date(2023, 6, 1), count=38,  cover_photo_url="https://s3.example.com/p3.jpg"),
]

timeline_router = APIRouter(prefix="/timeline", tags=["timeline"])


@timeline_router.get("/", response_model=list[MonthGroup])
async def get_timeline() -> list[MonthGroup]:
    """
    Return photos grouped by calendar month, newest first.

    SQL equivalent:
        SELECT date_trunc('month', taken_at) AS month,
               COUNT(*) AS count,
               (SELECT s3_key FROM photos p2
                WHERE date_trunc('month', p2.taken_at) = date_trunc('month', p.taken_at)
                ORDER BY taken_at DESC LIMIT 1) AS cover_key
        FROM photos p
        GROUP BY 1
        ORDER BY 1 DESC
    """
    return MOCK_TIMELINE


timeline_app = FastAPI()
timeline_app.include_router(timeline_router)
tc = TestClient(timeline_app)

resp = tc.get("/timeline/")
print(resp.status_code)
for row in resp.json():
    print(f"  {row['month']}  {row['count']} photos")

## People Albums

**Person cover thumbnails.** `GET /people/` returns persons sorted by photo count. Each person needs a face crop thumbnail — a small square image of their face, not the full photo — so the People screen looks like a contact list rather than a gallery of full-sized images. We generate these on demand:

1. Retrieve the `Face` record's bounding box `(x, y, w, h)` and the associated `Photo`'s `s3_key`.
2. Download the original photo from S3 into memory.
3. Crop the bounding box with Pillow, adding 20% padding on each side so the face isn't clipped at the edges.
4. Upload the crop to the `thumbnails/faces/{face_id}.jpg` S3 prefix.
5. Return a presigned URL for the thumbnail.

Step 4 happens once at indexing time (or lazily on first request); subsequent calls return the cached thumbnail URL.

**NOTE:** Bounding box coordinates from FaceNet are typically in pixel units relative to the original image. Always verify the coordinate system matches the Pillow crop call — some models return normalized coordinates in $[0, 1]$ that must be scaled by image width and height before cropping.

Defining `generate_face_thumbnail`: download, crop, upload, presign:

In [ ]:
import io
from dataclasses import dataclass
from unittest.mock import MagicMock

try:
    from PIL import Image as PILImage
    _PIL_AVAILABLE = True
except ImportError:
    _PIL_AVAILABLE = False


@dataclass
class Face:
    face_id:  str
    photo_id: str
    bbox_x:   int    # pixels, top-left x
    bbox_y:   int    # pixels, top-left y
    bbox_w:   int    # pixels, width
    bbox_h:   int    # pixels, height


@dataclass
class Photo:
    photo_id: str
    s3_key:   str
    width_px: int
    height_px: int


THUMBNAIL_BUCKET = "my-photo-app-thumbnails"
THUMBNAIL_PREFIX = "thumbnails/faces"
PADDING_FACTOR   = 0.20   # 20% padding around the face crop


def generate_face_thumbnail(
    face: Face,
    photo: Photo,
    s3_client,
    source_bucket: str,
    expiry_s: int = 3600,
) -> str:
    """
    Crop the face bounding box from the original S3 image, upload the
    thumbnail to THUMBNAIL_BUCKET, and return a presigned URL.
    """
    # 1. Download original image bytes
    obj = s3_client.get_object(Bucket=source_bucket, Key=photo.s3_key)
    img_bytes = obj["Body"].read()

    # 2. Crop with padding
    if _PIL_AVAILABLE:
        img = PILImage.open(io.BytesIO(img_bytes)).convert("RGB")
        pad_x = int(face.bbox_w * PADDING_FACTOR)
        pad_y = int(face.bbox_h * PADDING_FACTOR)
        left   = max(0, face.bbox_x - pad_x)
        top    = max(0, face.bbox_y - pad_y)
        right  = min(photo.width_px,  face.bbox_x + face.bbox_w + pad_x)
        bottom = min(photo.height_px, face.bbox_y + face.bbox_h + pad_y)
        crop   = img.crop((left, top, right, bottom))
        buf    = io.BytesIO()
        crop.save(buf, format="JPEG", quality=85)
        thumbnail_bytes = buf.getvalue()
    else:
        thumbnail_bytes = img_bytes   # fallback: upload original (demo only)

    # 3. Upload thumbnail
    thumb_key = f"{THUMBNAIL_PREFIX}/{face.face_id}.jpg"
    s3_client.put_object(
        Bucket=THUMBNAIL_BUCKET,
        Key=thumb_key,
        Body=thumbnail_bytes,
        ContentType="image/jpeg",
    )

    # 4. Presign
    url = s3_client.generate_presigned_url(
        "get_object",
        Params={"Bucket": THUMBNAIL_BUCKET, "Key": thumb_key},
        ExpiresIn=expiry_s,
    )
    return url


# Verify the function signature and logic with a mock S3 client
mock_s3 = MagicMock()
mock_s3.get_object.return_value = {"Body": MagicMock(read=lambda: b"fake-image-bytes")}
mock_s3.generate_presigned_url.return_value = "https://s3.example.com/thumb.jpg?presign=..."

face  = Face("face-001", "photo-001", 120, 80, 100, 100)
photo = Photo("photo-001", "photos/2023/07/img001.jpg", 1920, 1080)

url = generate_face_thumbnail(face, photo, mock_s3, source_bucket="my-photos")
print(f"Thumbnail URL: {url}")
mock_s3.put_object.assert_called_once()
print("put_object called ✓")

## Semantic Search Flow

**End-to-end latency.** The `POST /search/` endpoint chains three stages:

1. **CLIP text encode:** encode the query string into a $\mathbb{R}^{512}$ vector. On CPU, this takes approximately 20ms for a CLIP ViT-B/32 model.
2. **pgvector HNSW query:** `SELECT id, s3_key FROM photos ORDER BY embedding <=> $1 LIMIT $2`. With HNSW index at `ef_search=40`, this takes approximately 5ms for 100K photos.
3. **Presign batch:** generate one presigned S3 URL per result photo. This is pure Python with no I/O: each call to `s3_client.generate_presigned_url` is a local HMAC computation taking < 0.5ms. For a 20-result page, this adds < 10ms.

Total: well under 50ms end-to-end, making semantic search feel instantaneous to the user.

Full `POST /search/` endpoint with per-stage timing:

In [ ]:
import time
import numpy as np
from pydantic import BaseModel
from fastapi import APIRouter
from fastapi.testclient import TestClient
from fastapi import FastAPI


class SearchRequest(BaseModel):
    query: str
    top_k: int = 20


class SearchResult(BaseModel):
    photo_id:      str
    presigned_url: str
    score:         float


# Stubs for CLIP encoder and DB/S3 clients
def _clip_encode_text(text: str) -> np.ndarray:
    """Stub CLIP text encoder — returns a random unit vector."""
    rng = np.random.default_rng(abs(hash(text)) % (2**31))
    v = rng.standard_normal(512).astype(np.float32)
    return v / np.linalg.norm(v)


def _pgvector_search(embedding: np.ndarray, top_k: int) -> list[dict]:
    """Stub pgvector HNSW search — returns fake photo IDs with random scores."""
    rng = np.random.default_rng(0)
    return [
        {"photo_id": f"photo-{i:04d}", "s3_key": f"photos/img{i:04d}.jpg",
         "score": float(rng.uniform(0.7, 1.0))}
        for i in range(top_k)
    ]


def _batch_presign(s3_keys: list[str]) -> list[str]:
    """Stub presign — returns fake URLs."""
    return [f"https://s3.example.com/{k}?presign=..." for k in s3_keys]


search_router = APIRouter(prefix="/search", tags=["search"])


@search_router.post("/", response_model=list[SearchResult])
async def search_photos(req: SearchRequest) -> list[SearchResult]:
    timings: dict[str, float] = {}

    t0 = time.perf_counter()
    query_vec = _clip_encode_text(req.query)          # stage 1: CLIP encode
    timings["clip_ms"] = (time.perf_counter() - t0) * 1000

    t1 = time.perf_counter()
    rows = _pgvector_search(query_vec, req.top_k)     # stage 2: vector search
    timings["pgvector_ms"] = (time.perf_counter() - t1) * 1000

    t2 = time.perf_counter()
    s3_keys = [r["s3_key"] for r in rows]
    urls = _batch_presign(s3_keys)                    # stage 3: presign
    timings["presign_ms"] = (time.perf_counter() - t2) * 1000

    timings["total_ms"] = (time.perf_counter() - t0) * 1000
    print("Latency breakdown:", {k: f"{v:.2f}ms" for k, v in timings.items()})

    return [
        SearchResult(photo_id=r["photo_id"], presigned_url=url, score=r["score"])
        for r, url in zip(rows, urls)
    ]


search_app = FastAPI()
search_app.include_router(search_router)
sc = TestClient(search_app)

resp = sc.post("/search/", json={"query": "beach with children", "top_k": 5})
print(f"\nStatus: {resp.status_code}  Results: {len(resp.json())}")

## Query Planning Integration

**`POST /search/smart`.** The smart search endpoint chains the query planner from notebook 13 with the vector search from the previous section. The natural language query first passes through `parse_query`, which extracts a `SearchQuery` with optional `person_name`, `date_range`, and `location_hint` fields. These structured filters are applied as SQL `WHERE` clauses to restrict the candidate pool, and then CLIP vector ranking is applied over the filtered results only.

This two-stage approach is substantially more precise than pure vector search: a query like "photos of Alice at the beach in summer 2022" would otherwise return beach photos from all years and people, ranked only by visual similarity to "beach". Applying the person and date filter first ensures that only Alice's summer 2022 photos compete in the CLIP ranking.

Implementing `SmartSearchHandler` that chains `parse_query` → SQL filter → vector rank:

In [ ]:
import asyncio
import json
from datetime import date
from pydantic import BaseModel
from unittest.mock import AsyncMock, MagicMock


class SearchQuery(BaseModel):
    person_name:   str | None = None
    location_hint: str | None = None
    start_date:    date | None = None
    end_date:      date | None = None
    freeform:      str | None = None


class SmartSearchHandler:
    """Chains GPT-4o-mini query parsing with SQL filter + CLIP vector ranking."""

    def __init__(self, llm_client):
        self.llm = llm_client

    async def search(self, natural_language: str, top_k: int = 20) -> list[dict]:
        # Stage 1: parse query into structured filter
        sq = await self._parse(natural_language)
        print(f"Parsed filter: {sq.model_dump()}")

        # Stage 2: SQL filter (stub)
        candidates = self._sql_filter(sq)
        print(f"SQL filter: {len(candidates)} candidates")

        # Stage 3: CLIP rank over candidates
        query_text = sq.freeform or natural_language
        ranked = self._clip_rank(query_text, candidates, top_k)
        return ranked

    async def _parse(self, text: str) -> SearchQuery:
        resp = await self.llm.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Extract search filter as JSON."},
                {"role": "user",   "content": text},
            ],
            response_format={"type": "json_object"},
        )
        return SearchQuery.model_validate_json(resp.choices[0].message.content)

    def _sql_filter(self, sq: SearchQuery) -> list[str]:
        """Stub: return fake photo IDs matching the structured filter."""
        return [f"photo-{i:04d}" for i in range(50)]

    def _clip_rank(self, query: str, photo_ids: list[str], top_k: int) -> list[dict]:
        """Stub: return top_k IDs with random scores."""
        import random; random.seed(42)
        scored = [(pid, random.uniform(0.6, 1.0)) for pid in photo_ids]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [{"photo_id": pid, "score": round(s, 4)} for pid, s in scored[:top_k]]


# Mock LLM client returning a parsed filter
def _make_mock_client(response_text: str):
    choice = MagicMock()
    choice.message.content = response_text
    completion = MagicMock()
    completion.choices = [choice]
    client = MagicMock()
    client.chat.completions.create = AsyncMock(return_value=completion)
    return client


mock_llm = _make_mock_client(json.dumps({
    "person_name": "Alice", "location_hint": "beach",
    "start_date": "2022-06-01", "end_date": "2022-08-31",
    "freeform": "beach",
}))

handler = SmartSearchHandler(mock_llm)
results = asyncio.run(
    handler.search("photos of Alice at the beach in summer 2022", top_k=5)
)
print(f"\nTop results: {results}")

## Performance Tuning

**Connection pool sizing.** SQLAlchemy's `create_async_engine` maintains a pool of persistent database connections. Each concurrent API request that needs the database acquires a connection from the pool; if all connections are in use, the request waits. Setting `pool_size=10, max_overflow=20` allows up to 30 concurrent database connections, more than sufficient for a single-server deployment handling dozens of simultaneous requests:

```python
engine = create_async_engine(
    DATABASE_URL,
    pool_size=10,
    max_overflow=20,
    pool_timeout=30,       # seconds to wait for a connection before raising
    pool_recycle=1800,     # recycle connections older than 30 minutes
)
```

<br>

**N+1 queries.** The most common SQLAlchemy performance bug: loading a list of photos and then issuing a separate `SELECT` for each photo's faces. With 100 photos per page, this is 101 queries instead of 2. The fix is `selectinload`, which loads all related face records in a single additional query.

<br>

**pgvector `ef_search`.** The HNSW index has a query-time parameter `ef_search` that controls the search accuracy/speed trade-off. Higher values visit more nodes, improving recall but increasing latency. Setting `ef_search=40` gives > 95% recall in most benchmarks at 5ms for 100K vectors; setting `ef_search=200` approaches exact search at ~20ms. We expose this as a query parameter so power users can tune it.

:::{.callout-caution}
`SET hnsw.ef_search = 200` is a session-level setting in Postgres. In an async SQLAlchemy session that reuses connections from the pool, a session-level SET persists for the lifetime of the connection: subsequent requests that share the connection inherit the setting. Always reset after use or use a transaction-scoped SET.

:::

Demonstrating the N+1 query problem and the `selectinload` fix with a query counter:

In [ ]:
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class MockFace:
    face_id:    str
    person_id:  str


@dataclass
class MockPhoto:
    photo_id:    str
    _face_loader: Callable | None = field(default=None, repr=False)

    @property
    def faces(self) -> list[MockFace]:
        """Lazy loader — simulates SQLAlchemy lazy loading (N+1 trigger)."""
        if self._face_loader:
            return self._face_loader(self.photo_id)
        return []


# Simulated DB with query counter
query_count = 0

FACES_DB: dict[str, list[MockFace]] = {
    f"photo-{i:02d}": [MockFace(f"face-{i}-1", f"person-{i%3}")]
    for i in range(10)
}

def lazy_face_loader(photo_id: str) -> list[MockFace]:
    global query_count
    query_count += 1           # each call = one SQL query (the N+1 problem)
    return FACES_DB.get(photo_id, [])


# N+1 pattern: fetch photos, then access .faces on each
query_count = 0
photos = [MockPhoto(f"photo-{i:02d}", _face_loader=lazy_face_loader) for i in range(10)]
query_count += 1  # the initial 'SELECT * FROM photos' query

face_lists_n1 = [p.faces for p in photos]   # triggers 10 additional SELECT queries
print(f"N+1 pattern:   {query_count} queries for {len(photos)} photos")

# selectinload equivalent: load all faces in one extra query upfront
query_count = 0
query_count += 1  # SELECT * FROM photos
photo_ids = [f"photo-{i:02d}" for i in range(10)]
all_faces = {pid: FACES_DB.get(pid, []) for pid in photo_ids}  # one SELECT WHERE photo_id IN (...)
query_count += 1

photos_eager = [MockPhoto(pid, _face_loader=None) for pid in photo_ids]
print(f"selectinload:  {query_count} queries for {len(photos_eager)} photos  ← fixed")

## Reliability Patterns

**DB retry on startup.** When `docker-compose up` starts both the `api` and `db` containers simultaneously, the API process starts in milliseconds while Postgres takes several seconds to initialize. Without retry logic, the API fails immediately with `ConnectionRefusedError` and exits. We use `tenacity` to retry the connection with exponential backoff until Postgres is ready:

<br>

**Idempotent pipeline.** Re-triggering `POST /pipeline/start` on the same S3 prefix is safe because all writes use upsert semantics: photos and faces are inserted or updated by their content hash, never duplicated. A user who accidentally triggers indexing twice will see the same result set both times, with no phantom duplicates.

<br>

**Graceful shutdown.** FastAPI's lifespan context manager runs cleanup code on `SIGTERM` (the default Docker stop signal). We use it to close the SQLAlchemy connection pool and send a Close frame to all connected WebSocket clients, so they can reconnect cleanly rather than seeing an abrupt disconnect:

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    await engine.connect()         # startup
    yield
    await engine.dispose()         # shutdown: drain connection pool
    await manager.close_all()      # shutdown: close WebSocket connections
```

Implementing `retry_connect` with `tenacity`:

In [ ]:
import asyncio

try:
    from tenacity import (
        AsyncRetrying,
        stop_after_attempt,
        wait_exponential,
        retry_if_exception_type,
    )
    _TENACITY = True
except ImportError:
    _TENACITY = False


class ConnectionRefusedError(OSError):
    """Simulated asyncpg ConnectionRefusedError."""


async def retry_connect(dsn: str, max_attempts: int = 10) -> str:
    """
    Attempt to connect to the database with exponential backoff.
    Returns the DSN string on success (simulates returning the engine).
    """
    attempt_log: list[str] = []

    # Simulate Postgres being unavailable for the first 3 attempts
    calls = {"n": 0}

    async def _try_connect():
        calls["n"] += 1
        if calls["n"] < 4:
            attempt_log.append(f"attempt {calls['n']}: ConnectionRefusedError")
            raise ConnectionRefusedError("Connection refused")
        attempt_log.append(f"attempt {calls['n']}: connected")
        return dsn

    if _TENACITY:
        async for attempt in AsyncRetrying(
            stop=stop_after_attempt(max_attempts),
            wait=wait_exponential(multiplier=0.1, min=0.1, max=2),
            retry=retry_if_exception_type(ConnectionRefusedError),
            reraise=True,
        ):
            with attempt:
                result = await _try_connect()
    else:
        # Fallback: simple retry loop if tenacity not installed
        for i in range(max_attempts):
            try:
                result = await _try_connect()
                break
            except ConnectionRefusedError:
                if i == max_attempts - 1:
                    raise
                await asyncio.sleep(0.1 * (2 ** i))

    for log in attempt_log:
        print(log)
    return result


conn = asyncio.run(retry_connect("postgresql+asyncpg://user:pass@db:5432/photos"))
print(f"Connected: {conn}")

## Appendix: Scaling Beyond One Machine

**Horizontal API scaling.** Running multiple FastAPI instances behind a load balancer (nginx, HAProxy, or a cloud ALB) distributes traffic across processes and enables zero-downtime rolling deploys. The main complication is the in-memory `ConnectionManager`: it stores WebSocket subscriptions in a Python `dict`, which is local to one process. A client connected to instance A will not receive broadcasts sent from instance B. The fix is to move the pub/sub layer to **Redis pub/sub** — each instance publishes to a Redis channel, and all instances subscribe; every broadcast reaches every connected client regardless of which instance they are on.

<br>

**Embedding workers.** CLIP and FaceNet inference are CPU-intensive and do not benefit from asyncio's cooperative multitasking, blocking the event loop. Moving them to separate worker processes (using `concurrent.futures.ProcessPoolExecutor` or dedicated `multiprocessing.Queue`-based workers) frees the API event loop for I/O while the CPU-bound work runs in parallel.

<br>

**S3 event notifications → SQS.** The current pipeline polls S3 on demand when the user presses "Index". At larger scale, new photos should trigger indexing automatically. S3 event notifications can publish `s3:ObjectCreated:*` events to an SQS queue; a pool of worker processes consumes the queue and processes each photo independently. This fully decouples ingestion from the API server (the API no longer runs the pipeline at all) and scales the indexing throughput horizontally by adding more workers.

:::{.callout-note}
With SQS-driven workers, the `ConnectionManager` WebSocket broadcasts still need to flow from the workers back to the API (and then to the Flet clients). A Redis pub/sub channel serves double duty here: workers publish progress events; API instances subscribe and forward to their connected WebSocket clients.

:::

---

■